In [40]:
import matplotlib.pyplot as plt
import numpy as np
import time
from time import perf_counter
from timeit import repeat
import gc

from numpy.polynomial.legendre import legval
from scipy.special import eval_legendre,factorial
from scipy.optimize import least_squares

# Import des données

In [2]:
# Tableau : He-HCN
He = np.genfromtxt("Data/He-HCN.txt",
                   delimiter=",", names=True, dtype=None, encoding="utf-8")

# Optimisation des fonctions

Dans ce Notebook, on s'intéresse à l'optimisation des performances dans l'implémentation de notre fonction :
$$V(R,\theta)=V_{sh}(R,\theta)+V_{as}(R,\theta)$$

Il semble important de passer un peu de temps dessus ces fonctions seront appelées de nombreuses fois pour l'estimation des paramètres optimaux.

## Premier jet

In [3]:
def G(R,Theta,g0,g1,g2,g3):
  g = 0.
  for i in range(6):
    g = g + (g0[i]+g1[i]*R+g2[i]*R*R+g3[i]*R*R*R)*eval_legendre(i,np.cos(Theta))
  return g

In [4]:
def X(Theta,x):
  y = 0.
  for i in range(6):
    y = y + x[i]*eval_legendre(i,np.cos(Theta))
  return y

In [5]:
def f(n,x):
  y = 0.
  for k in range(n+1):
    y = y + (x**k)/factorial(k)
  y = 1. - np.exp(-x)*y
  return y

In [6]:
def V(vars, *params):

  R, Theta = vars

  # Conversion des degrés en radian
  Theta_rad = np.deg2rad(Theta)

  #  Récupération des paramètres
  b  = params[0:6]    # b0, b1, b2, b3, b4, b5
  c  = params[6:10]   # c0, c1, c2, c3
  d  = params[10:16]  # d0, d1, d2, d3, d4, d5
  g0 = params[16:22]  # g00, g01, g02, g03, g04, g05
  g1 = params[22:28]  # g10, g11, g12, g13, g14, g15
  g2 = params[28:34]  # g20, g21, g22, g23, g24, g25
  g3 = params[34:40]  # g30, g31, g32, g33, g34, g35

  # Variables utiles
  abs_BR = np.abs(X(Theta_rad,b)*R)
  cosTh = np.cos(Theta_rad)

  # Short-Range
  V_sh = G(R,Theta_rad,g0,g1,g2,g3) * np.exp(X(Theta_rad,d)-X(Theta_rad,b)*R)

  # Asymptotyhic
  V_as = f(6,abs_BR) * (c[0]*eval_legendre(0,cosTh)+c[2]*eval_legendre(2,cosTh))/(R**6)\
       + f(7,abs_BR) * (c[1]*eval_legendre(1,cosTh)+c[3]*eval_legendre(3,cosTh))/(R**7)

  return V_sh + V_as

## Versions optimisées

**Méthodologie :**

Après avoir implémenté les différentes fonctions qui sont présentées dans l'article "Theoretical study of the He-HCN,..., complexes".

Je me sert de différents LLM pour trouver des pistes d'améliorations des performances de mon premier programme.

Avec des prompts du style "Quel est le plus performant entre telle chose et telle chose" ou "Comment puis je améliorer les performances" etc.

Liste des points à améliorer :

1.   Certaines variables sont recalculées plusieurs fois de manière inutile
2.   Vectorisation de certaines fonctions, éviter l'appel de fonctions exterieur lorsque ce n'est pas nécessaire
3.   Utilisation de fonctions numpy comme legval lorsque c'est utile



In [7]:
def f6_opt(x):
  y = 1.0 + x * (1.0 + x * (0.5 + x * (1.0/6.0 + x * (1.0/24.0 + x * (1.0/120.0 + x / 720.0)))))
  return 1.0 - np.exp(-x) * y

def f7_opt(x):
  y = 1.0 + x * (1.0 + x * (0.5 + x * (1.0/6.0 + x * (1.0/24.0 + x * (1.0/120.0 + x * (1.0/720.0 + x / 5040.0))))))
  return 1.0 - np.exp(-x) * y

In [121]:
def V_opt(p, R, Theta):
  R = np.asarray(R, dtype=float)
  # On calcule ces valeurs car utiles plusieurs fois
  cosTh = np.cos(np.deg2rad(np.asarray(Theta, dtype=float)))

  # Paramètres
  b  = p[0:6]                                        # b0, b1, b2, b3, b4, b5
  c  = p[6:10]                                       # c0, c1, c2, c3
  d  = p[10:16]                                      # d0, d1, d2, d3, d4, d5
  g0 = np.asarray(p[16:22], dtype=float).reshape(6,) # g00, g01, g02, g03, g04, g05
  g1 = np.asarray(p[22:28], dtype=float).reshape(6,) # g10, g11, g12, g13, g14, g15
  g2 = np.asarray(p[28:34], dtype=float).reshape(6,) # g20, g21, g22, g23, g24, g25
  g3 = np.asarray(p[34:40], dtype=float).reshape(6,) # g30, g31, g32, g33, g34, g35

  # X(theta) ------------------------------------ #
  # --------------------------------------------- #
  X_b = legval(cosTh, b)
  X_d = legval(cosTh, d)

  X_bR = X_b * R
  abs_BR = np.abs(X_bR)
  # --------------------------------------------- #

  # G(R,theta) ---------------------------------- #
  # --------------------------------------------- #
  g = g0[:, None] + R * (g1[:, None] + R * (g2[:, None] + R * g3[:, None]))
  Pl = eval_legendre(np.arange(6)[:,None],cosTh)
  # G = np.sum(g * Pl, axis=0)
  # ------------------------------------------- #

  # Short-Range
  V_sh = np.sum(g * Pl, axis=0) * np.exp(X_d-X_bR)

  # Asymptotic
  inv_R = 1.0/R
  inv_R6 = inv_R * inv_R * inv_R * inv_R * inv_R * inv_R
  inv_R7 = inv_R6 * inv_R

  V_as = (f6_opt(abs_BR) * legval(cosTh,[c[0],0,c[2]]) * inv_R6
        + f7_opt(abs_BR) * legval(cosTh,[0,c[1],0,c[3]]) * inv_R7)

  return V_sh + V_as

In [91]:
def V_opt2(p, R, Theta):
    """
    p     : vecteur de taille 40
    R     : vecteur de taille N
    Theta : vecteur de taille N
    """
    
    R = np.asarray(R, dtype=float)
    # On calcule ces valeurs car utiles plusieurs fois
    cosTh = np.cos(np.deg2rad(np.asarray(Theta, dtype=float)))
    
    # Paramètres
    b  = p[0:6]                                  # b0, b1, b2, b3, b4, b5
    c  = p[6:10]                                 # c0, c1, c2, c3
    d  = p[10:16]                                # d0, d1, d2, d3, d4, d5
    g0 = np.asarray(p[16:22], dtype=float)       # g00, g01, g02, g03, g04, g05
    g1 = np.asarray(p[22:28], dtype=float)       # g10, g11, g12, g13, g14, g15
    g2 = np.asarray(p[28:34], dtype=float)       # g20, g21, g22, g23, g24, g25
    g3 = np.asarray(p[34:40], dtype=float)       # g30, g31, g32, g33, g34, g35
    
    # ----------------- X(theta) ------------------ #
    # --------------------------------------------- #
    X_b = legval(cosTh, b)                          # ndarray -> shape (N,)
    X_d = legval(cosTh, d)                          # ndarray -> shape (N,)
    
    X_bR = X_b * R                                  # ndarray -> shape (N,)
    abs_BR = np.abs(X_bR)                           # ndarray -> shape (N,)
    # --------------------------------------------- #

    
    # ----------------- G(R,theta) ---------------- #
    # --------------------------------------------- #

    # Pl : ndarray -> shape (6,N)
    Pl = eval_legendre(np.arange(6)[:,None],cosTh)
    
    # g : ndarray -> shape (6,1)
    g = g0[:, None] + R * (g1[:, None] + R * (g2[:, None] + R *  g3[:, None]))            
    
    
    # mult = g * Pl  -> g[i,0] * Pl[i,x]  (x=0,..N-1 ; i=0,..5)
    # mult : ndarray -> shape (6,N)
    #
    # G = np.sum(mult, axis=0) -> axis=0 : pour chaque colonne x, 
    #                             on somme tous les éléments entre eux
    # G : ndarray -> shape (N,)
    # --------------------------------------------- #


    
    # ---------------- Short-Range ---------------- #
    
    V_sh = np.sum(g * Pl, axis=0) * np.exp(np.clip(X_d-X_bR,-200,200))
    # Bloquage des valeurs trop petites/grandes avec np.clip
    # V_sh = G * exp(...)
    # V_sh : ndarray -> shape (N,)
    # --------------------------------------------- #

    
    # ---------------- Asymptotic ----------------- #
    inv_R = 1.0/R
    inv_R6 = inv_R * inv_R * inv_R * inv_R * inv_R * inv_R
    inv_R7 = inv_R6 * inv_R
    
    V_as = (f6_opt(abs_BR) * legval(cosTh,[c[0],0,c[2]]) * inv_R6
          + f7_opt(abs_BR) * legval(cosTh,[0,c[1],0,c[3]]) * inv_R7)

    # --------------------------------------------- #
    
    V_total = V_sh + V_as # ndarray -> shape (N,)
    
    # Pour ne pas renvoyer NaN en cas d'erreur
    V_total = np.nan_to_num(V_total, nan=1e100, posinf=1e100, neginf=-1e100)
    # Bloquage des valeurs trop grandes
    V_total = np.clip(V_total, -1e100, 1e100)
      
    return V_total # Energie renvoyée en mEh (selon les parametres p donnés)

Les fonctions V_opt et V_opt2 sont très similaires. V_opt2 contient des sécurités pour éviter les problèmes d'overflow ce qui aura surement un impact sur les performances. Ce compromis peut être important pour la suite lors du calcul des paramètres p optimaux. 

## Contrôle des fonctions

On s'assure ensuite que les différentes fonctions donnent des résultats numériques suffisament proches pour affirmer qu'elles reproduisent le même modèle

In [122]:
np.random.seed(0)

R = np.random.uniform(2.0, 10.0, 500)
Theta = np.random.uniform(0.0, 180.0, 500)

p = np.random.randn(40)

V1 = V((R, Theta), *p)
V2 = V_opt(p, R, Theta)
V3 = V_opt2(p, R, Theta)

diff1 = V1 - V2
diff2 = V1 - V3

print("max  diff 1 :", np.max(np.abs(diff1)))
print("mean diff 1 :", np.mean(np.abs(diff1)))
print("std  diff 1 :", np.std(diff1))
print("max  diff 2 :", np.max(np.abs(diff2)))
print("mean diff 2 :", np.mean(np.abs(diff2)))
print("std  diff 2 :", np.std(diff2))

max  diff 1 : 1.7881393432617188e-06
mean diff 1 : 1.2202130886608412e-08
std  diff 1 : 1.09020624706844e-07
max  diff 2 : 1.7881393432617188e-06
mean diff 2 : 1.2202130886608412e-08
std  diff 2 : 1.09020624706844e-07


# Tests de performances

## Fonctions pures

In [162]:
R      = np.linspace(4.5,10,10000)
Theta  = np.linspace(0,180,10000)

In [158]:
def bench(func, p, R, Theta, n=100):
    # warm-up (cache CPU + Python)
    for _ in range(5):
        func(p, R, Theta)

    gc.collect()

    start = perf_counter()
    for _ in range(n):
        func(p, R, Theta)
    return (perf_counter() - start) / n

In [165]:
params = np.random.rand(40)

t1 = bench(lambda p, R, T: V((R, T), *p), params, R, Theta)
t2 = bench(V_opt, params, R, Theta)
t3 = bench(V_opt2, params, R, Theta)
t4 = bench(V_opt3, params, R, Theta)

print(f"V      : {t1:.6e} s/appel")
print(f"V_opt  : {t2:.6e} s/appel")
print(f"V_opt2 : {t3:.6e} s/appel")

V      : 3.356061e-03 s/appel
V_opt  : 1.114889e-03 s/appel
V_opt2 : 1.118117e-03 s/appel


In [135]:
print("Speedup V_opt  :", t1 / t2)
print("Speedup V_opt2 :", t1 / t3)
print("Speedup opt2 vs opt1 :", t2 / t3)

Speedup V_opt  : 3.1458464088378775
Speedup V_opt2 : 2.7380626916980653
Speedup opt2 vs opt1 : 0.8703739267135888


In [142]:
v1 = V((R, Theta), *params_init)
v2 = V_opt(params_init, R, Theta)
v3 = V_opt2(params_init, R, Theta)

print(np.max(np.abs(v1 - v2)))
print(np.max(np.abs(v1 - v3)))

6.934897101018578e-12
6.934897101018578e-12


## Residus

In [111]:
def residus_1(params, R, Theta, V_tableau):
    return  V((R, Theta), *params) - V_tableau

def residus_2(p, R,Theta, V_tableau):
    return V_opt(p, R, Theta) - V_tableau

def residus_3(p, R,Theta, V_tableau):
    return V_opt2(p, R, Theta) - V_tableau

In [112]:
R = He["R"]
Theta = He["Theta"]
V_tableau = He["Energy"]

params_init = np.random.rand(40)
params_init

array([4.75324782e-01, 9.69205872e-01, 2.65632548e-01, 1.35087066e-02,
       4.83752865e-01, 2.56113795e-01, 8.23717672e-01, 2.32772672e-01,
       3.10629218e-01, 7.91227431e-01, 7.15143252e-01, 5.58051237e-01,
       7.04948062e-01, 4.18636864e-01, 5.31004761e-03, 1.13551285e-02,
       5.11221788e-01, 8.32909797e-02, 5.10754802e-02, 9.65516639e-01,
       8.59002640e-01, 1.52027227e-01, 6.64218590e-04, 9.41667795e-01,
       2.78325298e-01, 1.85897603e-01, 6.91508108e-01, 1.08903739e-01,
       2.64649598e-01, 9.75094680e-01, 6.39462774e-01, 5.20677791e-01,
       3.97918615e-01, 7.74500955e-01, 1.40957477e-01, 9.67337802e-01,
       8.61123008e-01, 6.17656983e-01, 4.29061904e-02, 7.00855649e-01])

## Cout d'une évaluation

In [116]:
n = 100

t1 = min(repeat(
    lambda: residus_1(params_init, R, Theta, V_tableau),
    number=n,
    repeat=5
))

t2 = min(repeat(
    lambda: residus_2(params_init, R, Theta, V_tableau),
    number=n,
    repeat=5
))

t3 = min(repeat(
    lambda: residus_3(params_init, R, Theta, V_tableau),
    number=n,
    repeat=5
))

print(f"residus_1 : {t1/n:.3e} s/appel")
print(f"residus_2 : {t2/n:.3e} s/appel")
print(f"residus_3 : {t3/n:.3e} s/appel")

residus_1 : 2.526e-04 s/appel
residus_2 : 1.010e-04 s/appel
residus_3 : 1.093e-04 s/appel


## Performances pour least_squares

In [19]:
# On appelle les deux fonctions une première fois pour retirer les couts liés à
# l'initialisation des fonctions
residus_1(params_init, R, Theta, V_tableau);
residus_2(params_init, R, Theta, V_tableau);
residus_3(params_init, R, Theta, V_tableau);

N = 10
temps1 = []
temps2 = []
temps3 = []

for _ in range(N):
    p0 = np.random.rand(40)

    start = time.perf_counter()
    least_squares(residus_1, p0, args=(R, Theta, V_tableau))
    temps1.append(time.perf_counter() - start)

    start = time.perf_counter()
    least_squares(residus_2, p0, args=(R, Theta, V_tableau))
    temps2.append(time.perf_counter() - start)

    start = time.perf_counter()
    least_squares(residus_3, p0, args=(R, Theta, V_tableau))
    temps3.append(time.perf_counter() - start)

Méthode 1
  moyenne = 11.024s
  std     = 3.593s
Méthode 2
  moyenne = 4.845s
  std     = 1.371s
Méthode 3
  moyenne = 5.490s
  std     = 1.552s
Comparaison du nombre d'évaluations de least_squares:
Méthode 1 531
Méthode 2 532
Méthode 3 532


In [117]:
print("Méthode 1")
print(f"  moyenne = {np.mean(temps1):.3f}s")
print(f"  std     = {np.std(temps1):.3f}s")
print("Méthode 2")
print(f"  moyenne = {np.mean(temps2):.3f}s")
print(f"  std     = {np.std(temps2):.3f}s")
print("Méthode 3")
print(f"  moyenne = {np.mean(temps3):.3f}s")
print(f"  std     = {np.std(temps3):.3f}s")

print("Comparaison du nombre d'évaluations de least_squares:")
print("Méthode 1",resol1.nfev)
print("Méthode 2",resol2.nfev)
print("Méthode 3",resol3.nfev)

Méthode 1
  moyenne = 11.024s
  std     = 3.593s
Méthode 2
  moyenne = 4.845s
  std     = 1.371s
Méthode 3
  moyenne = 5.490s
  std     = 1.552s
Comparaison du nombre d'évaluations de least_squares:
Méthode 1 531
Méthode 2 532
Méthode 3 532


In [118]:
print(resol1.cost)
print(resol2.cost)
print(resol3.cost)

39684.62436716206
39684.624273261266
39684.624273261266


In [119]:
print(np.linalg.norm(resol1.fun))
print(np.linalg.norm(resol2.fun))
print(np.linalg.norm(resol3.fun))

281.7254847086506
281.7254843753446
281.7254843753446


In [120]:
r1 = residus_1(params_init, R, Theta, V_tableau)
r2 = residus_2(params_init, R, Theta, V_tableau)
r3 = residus_3(params_init, R, Theta, V_tableau)

print(np.max(np.abs(r1-r2)))
print(np.max(np.abs(r1-r3)))

8.753886504564434e-12
8.753886504564434e-12


In [28]:
print(f"Speedup opt1 : {np.mean(temps1)/np.mean(temps2):.2f}x")
print(f"Speedup opt2 : {np.mean(temps1)/np.mean(temps3):.2f}x")

Speedup opt1 : 2.28x
Speedup opt2 : 2.01x


In [31]:
r1 = residus_1(params_init, R, Theta, V_tableau)
r2 = residus_2(params_init, R, Theta, V_tableau)
r3 = residus_3(params_init, R, Theta, V_tableau)

print("max |r1-r2| =", np.max(np.abs(r1-r2)))
print("max |r1-r3| =", np.max(np.abs(r1-r3)))
print("max |r2-r3| =", np.max(np.abs(r2-r3)))

print(np.allclose(r1, r2))
print(np.allclose(r1, r3))

max |r1-r2| = 8.753886504564434e-12
max |r1-r3| = 8.753886504564434e-12
max |r2-r3| = 0.0
True
True


In [144]:
%load_ext line_profiler

In [147]:
%lprun -f V_opt2 V_opt2(p, R, Theta)

Timer unit: 1e-09 s

Total time: 0.00269612 s
File: /tmp/ipykernel_78924/274868062.py
Function: V_opt2 at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def V_opt2(p, R, Theta):
     2                                               """
     3                                               p     : vecteur de taille 40
     4                                               R     : vecteur de taille N
     5                                               Theta : vecteur de taille N
     6                                               """
     7                                           
     8         1       3140.0   3140.0      0.1      R = np.asarray(R, dtype=float)
     9                                               # On calcule ces valeurs car utiles plusieurs fois
    10         1     190238.0 190238.0      7.1      cosTh = np.cos(np.deg2rad(np.asarray(Theta, dtype=float)))
    11                                   